# RAG & LLM Application Development

---

## Learning Objectives
- Understand what RAG is and why it matters
- Build a complete RAG pipeline: ingest → chunk → embed → store → retrieve → generate
- Use LangChain to orchestrate the pipeline
- Add conversation memory to a chatbot
- Understand production UI options: Streamlit, FastAPI, Gradio

---
## Setup — Install Packages

In [1]:
# Install only what we need — lightweight and stable packages
# This cell takes about 1–2 minutes on first run
import subprocess, sys

pkgs = [
    "langchain==0.1.20",
    "langchain-community==0.0.38",
    "faiss-cpu",
    "sentence-transformers==2.7.0",
    "tiktoken",
]
for p in pkgs:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", p])

print("✓ All packages installed")

✓ All packages installed


In [2]:
# Core imports — all from stable, tested packages
import warnings, os, textwrap, numpy as np
warnings.filterwarnings("ignore")

from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.schema        import Document
from langchain.vectorstores  import FAISS
from langchain.embeddings    import HuggingFaceEmbeddings
from langchain.chains        import RetrievalQA, ConversationalRetrievalChain
from langchain.memory        import ConversationBufferMemory
from langchain.prompts       import PromptTemplate
from langchain.llms.base     import LLM
from typing                  import Optional, List

print("✓ All imports successful")

✓ All imports successful


---
# PART 1 — What is RAG and Why is it Powerful?

## The Problem with Plain LLMs

A regular LLM only knows what it learned during training. This creates real problems:

| Problem | Example |
|---------|--------|
| **Hallucination** | Confidently states wrong facts |
| **Outdated knowledge** | Does not know events after training cutoff |
| **No private data** | Cannot answer questions about YOUR company's documents |
| **No source attribution** | Cannot tell you WHERE the answer came from |

## What RAG Does

**RAG = Retrieval-Augmented Generation**

Instead of relying only on what the LLM memorised, RAG gives the LLM a **reference library** to look up at query time:

```
User Question
     ↓
Search your documents (vector similarity search)
     ↓
Find the most relevant chunks
     ↓
Give chunks + question to LLM
     ↓
Grounded, cited answer
```

---
# The RAG Pipeline (Step by Step)

We build each step one cell at a time so you can explain each one to your students.

```
Step 1 — INGEST   : Load raw documents
Step 2 — CHUNK    : Split into small overlapping pieces
Step 3 — EMBED    : Convert each chunk to a vector (numbers capturing meaning)
Step 4 — STORE    : Save vectors in FAISS (local vector database)
Step 5 — RETRIEVE : For a user question, find the most similar chunks
Step 6 — GENERATE : Give the chunks + question to the LLM → grounded answer
```

## Step 1 — Document Ingestion

In production you would load PDFs, websites, or Word documents.  
Here we create our knowledge base directly in the notebook — **no file downloads needed**.

LangChain wraps every piece of text in a **`Document`** object with:
- `page_content` — the actual text
- `metadata` — extra info (source file, topic, date, etc.)

In [3]:
# Our knowledge base — 6 documents about AI topics
# In real use, these would come from PDFs, web pages, or databases

raw_documents = [
    Document(
        page_content=(
            "Retrieval-Augmented Generation (RAG) is a technique that combines a "
            "retrieval system with a generative language model. Instead of relying "
            "solely on the model training data, RAG retrieves relevant documents "
            "from an external knowledge base and uses them as context for generation. "
            "This reduces hallucinations and keeps responses grounded in real data. "
            "RAG is especially useful when knowledge needs frequent updates "
            "or when working with private, domain-specific information."
        ),
        metadata={"source": "rag_overview.txt", "topic": "RAG"}
    ),
    Document(
        page_content=(
            "Vector embeddings are numerical representations of text that capture "
            "semantic meaning. A sentence is converted into a high-dimensional vector "
            "of numbers, typically 384 dimensions. Sentences with similar meanings "
            "produce vectors that are close to each other in this space, measured by "
            "cosine similarity. Popular embedding models include OpenAI text-embedding "
            "and Sentence Transformers all-MiniLM-L6-v2. Embeddings enable semantic "
            "search: finding documents by meaning, not just keyword matching."
        ),
        metadata={"source": "embeddings.txt", "topic": "Embeddings"}
    ),
    Document(
        page_content=(
            "A vector database stores and indexes vector embeddings for fast similarity search. "
            "Given a query vector, it finds the k most similar vectors very quickly. "
            "FAISS from Facebook AI is a popular free local option. "
            "Pinecone and Weaviate are managed cloud services. "
            "Chroma is simple and great for prototypes. "
            "Choosing the right vector database depends on scale, cost, "
            "and whether you need cloud management or local control."
        ),
        metadata={"source": "vector_databases.txt", "topic": "Vector DB"}
    ),
    Document(
        page_content=(
            "LangChain is an open-source Python framework for building LLM-powered applications. "
            "It provides document loaders to load PDFs, web pages, and databases. "
            "Text splitters chunk documents intelligently. "
            "Vector store wrappers interface with FAISS, Pinecone, and Chroma. "
            "Chains compose retriever plus LLM plus prompt into one workflow. "
            "Memory stores conversation history across multiple turns. "
            "LangChain reduces the code needed to build production RAG pipelines."
        ),
        metadata={"source": "langchain.txt", "topic": "LangChain"}
    ),
    Document(
        page_content=(
            "Hallucination in LLMs means the model generates confident but factually wrong statements. "
            "It happens because LLMs are trained to produce fluent text, not necessarily true text. "
            "RAG reduces hallucination by grounding responses in retrieved documents. "
            "Good prompt engineering also helps: instruct the model to say I do not know "
            "when the answer is not found in the provided context."
        ),
        metadata={"source": "hallucinations.txt", "topic": "Hallucinations"}
    ),
    Document(
        page_content=(
            "Streamlit is a Python framework for building interactive web applications quickly. "
            "It is widely used for LLM chatbot interfaces and data dashboards. "
            "Key features include st.chat_message for chat UI and st.session_state for history. "
            "Gradio is similar but focused on ML demos and sharing public links instantly. "
            "FastAPI is the choice for production REST APIs because it is fast and auto-documented. "
            "For teaching and demos use Gradio. For internal tools use Streamlit. "
            "For production APIs use FastAPI."
        ),
        metadata={"source": "ui_frameworks.txt", "topic": "UI Frameworks"}
    ),
]

print(f"Loaded {len(raw_documents)} documents\n")
for doc in raw_documents:
    print(f"  [{doc.metadata['topic']:<15}]  {doc.metadata['source']}")

Loaded 6 documents

  [RAG            ]  rag_overview.txt
  [Embeddings     ]  embeddings.txt
  [Vector DB      ]  vector_databases.txt
  [LangChain      ]  langchain.txt
  [Hallucinations ]  hallucinations.txt
  [UI Frameworks  ]  ui_frameworks.txt


## Step 2 — Document Chunking

**Why chunk?** LLMs have a limited context window — they can only read so much text at once. We split documents into small pieces.

Key settings:
- `chunk_size` — maximum characters per chunk (400–800 is typical)
- `chunk_overlap` — characters shared between consecutive chunks, to avoid losing context at boundaries

`RecursiveCharacterTextSplitter` is the recommended default — it tries paragraph breaks first, then sentences, keeping meaning intact.

In [4]:
# Split documents into chunks
# RecursiveCharacterTextSplitter tries to split at paragraph and sentence
# boundaries first — much better than just cutting at a fixed character count

splitter = RecursiveCharacterTextSplitter(
    chunk_size=400,     # max characters per chunk
    chunk_overlap=50,   # characters shared between consecutive chunks
)

chunks = splitter.split_documents(raw_documents)

print(f"Documents : {len(raw_documents)}")
print(f"Chunks    : {len(chunks)}")
print()
print("First chunk example:")
print("─" * 55)
print(f"Source : {chunks[0].metadata['source']}")
print(f"Length : {len(chunks[0].page_content)} chars")
print(f"Text   : {chunks[0].page_content}")

Documents : 6
Chunks    : 11

First chunk example:
───────────────────────────────────────────────────────
Source : rag_overview.txt
Length : 395 chars
Text   : Retrieval-Augmented Generation (RAG) is a technique that combines a retrieval system with a generative language model. Instead of relying solely on the model training data, RAG retrieves relevant documents from an external knowledge base and uses them as context for generation. This reduces hallucinations and keeps responses grounded in real data. RAG is especially useful when knowledge needs


## Step 3 — Vector Embeddings

Each chunk is converted to a vector — a list of numbers that captures its **meaning**.

```
"machine learning methods"  →  [0.21, -0.45, 0.88, ...]  384 numbers
"ML techniques"             →  [0.19, -0.43, 0.87, ...]  ← very similar!
"pizza recipe"              →  [-0.72, 0.11, -0.34, ...] ← very different
```

We use **`all-MiniLM-L6-v2`** — completely free, runs locally, downloads once (~90MB).

In [5]:
# Load the embedding model
# HuggingFaceEmbeddings from langchain.embeddings is the stable import
# Downloads ~90MB once and caches locally — fast on every run after that

print("Loading embedding model (first run downloads ~90MB, then cached)...")

embeddings = HuggingFaceEmbeddings(
    model_name="all-MiniLM-L6-v2",
    model_kwargs={"device": "cpu"}
)

# Quick test — embed one sentence
test_vec = embeddings.embed_query("What is RAG?")

print(f"\n✓ Embedding model loaded")
print(f"  Dimensions : {len(test_vec)}")
print(f"  First 6 values : {[round(v, 4) for v in test_vec[:6]]}...")
print(f"  Each number encodes a small aspect of the text meaning.")

Loading embedding model (first run downloads ~90MB, then cached)...

✓ Embedding model loaded
  Dimensions : 384
  First 6 values : [-0.0696, 0.0952, 0.016, 0.0068, -0.0884, 0.0142]...
  Each number encodes a small aspect of the text meaning.


In [6]:
# Demonstrate semantic similarity — why embeddings matter for search
# Cosine similarity: 1.0 = identical meaning, 0.0 = completely unrelated

def cosine_sim(a, b):
    a, b = np.array(a), np.array(b)
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))

sentences = [
    "RAG retrieves external documents to ground LLM responses",
    "Retrieval augmented generation fetches relevant text for the model",
    "Vector databases store embeddings for fast similarity search",
    "Pizza is a popular dish with tomato sauce and cheese",
]

vecs = [embeddings.embed_query(s) for s in sentences]

print(f"Reference sentence:\n  '{sentences[0]}'\n")
print(f"{'Comparison sentence':<52} {'Similarity':>10}")
print("─" * 65)
for i in range(1, len(sentences)):
    sim = cosine_sim(vecs[0], vecs[i])
    bar = "█" * int(sim * 20)
    print(f"{sentences[i]:<52} {sim:>8.3f}  {bar}")

print()
print("Sentences 1 and 2 have high similarity — both are about RAG.")
print("The pizza sentence has low similarity — correctly identified as unrelated.")

Reference sentence:
  'RAG retrieves external documents to ground LLM responses'

Comparison sentence                                  Similarity
─────────────────────────────────────────────────────────────────
Retrieval augmented generation fetches relevant text for the model    0.300  █████
Vector databases store embeddings for fast similarity search    0.124  ██
Pizza is a popular dish with tomato sauce and cheese   -0.082  

Sentences 1 and 2 have high similarity — both are about RAG.
The pizza sentence has low similarity — correctly identified as unrelated.


## Step 4 — Vector Store (FAISS)

**FAISS** is a free, local vector database from Facebook AI.  
`FAISS.from_documents()` embeds every chunk and builds a searchable index in one step.

In [7]:
# Build the FAISS vector store from our chunks
# This embeds every chunk and indexes them for instant similarity search

print(f"Building vector store from {len(chunks)} chunks...")

vectorstore = FAISS.from_documents(chunks, embeddings)

# Save to disk so we can reload without rebuilding
vectorstore.save_local("faiss_index")

print(f"\n✓ Vector store built and saved to ./faiss_index/")
print(f"  Indexed {len(chunks)} chunks as {len(test_vec)}-dimensional vectors")
print(f"  To reload later: FAISS.load_local('faiss_index', embeddings)")

Building vector store from 11 chunks...

✓ Vector store built and saved to ./faiss_index/
  Indexed 11 chunks as 384-dimensional vectors
  To reload later: FAISS.load_local('faiss_index', embeddings)


## Step 5 — Semantic Retrieval

When a user asks a question:
1. The question is embedded using the **same model** used for documents
2. FAISS finds the chunks whose vectors are **closest** to the question vector
3. Those top-k chunks are returned as context for the LLM

In [8]:
# Test retrieval manually before connecting it to a LLM
# similarity_search() embeds the query and returns the k most similar chunks

def show_retrieval(query, k=3):
    print(f"Query : '{query}'")
    print("=" * 60)
    results = vectorstore.similarity_search(query, k=k)
    for i, doc in enumerate(results):
        print(f"\nResult {i+1}  [Source: {doc.metadata['source']}]")
        print("─" * 40)
        print(textwrap.fill(doc.page_content, width=68))
    return results

_ = show_retrieval("How does RAG reduce hallucinations?")

Query : 'How does RAG reduce hallucinations?'

Result 1  [Source: hallucinations.txt]
────────────────────────────────────────
Hallucination in LLMs means the model generates confident but
factually wrong statements. It happens because LLMs are trained to
produce fluent text, not necessarily true text. RAG reduces
hallucination by grounding responses in retrieved documents. Good
prompt engineering also helps: instruct the model to say I do not
know when the answer is not found in the provided context.

Result 2  [Source: rag_overview.txt]
────────────────────────────────────────
RAG is especially useful when knowledge needs frequent updates or
when working with private, domain-specific information.

Result 3  [Source: rag_overview.txt]
────────────────────────────────────────
Retrieval-Augmented Generation (RAG) is a technique that combines a
retrieval system with a generative language model. Instead of
relying solely on the model training data, RAG retrieves relevant
documents from an

In [9]:
# Test with a question about vector databases
_ = show_retrieval("What vector database should I use?")

Query : 'What vector database should I use?'

Result 1  [Source: vector_databases.txt]
────────────────────────────────────────
A vector database stores and indexes vector embeddings for fast
similarity search. Given a query vector, it finds the k most similar
vectors very quickly. FAISS from Facebook AI is a popular free local
option. Pinecone and Weaviate are managed cloud services. Chroma is
simple and great for prototypes. Choosing the right vector database
depends on scale, cost, and whether you need cloud management

Result 2  [Source: langchain.txt]
────────────────────────────────────────
LangChain is an open-source Python framework for building LLM-
powered applications. It provides document loaders to load PDFs, web
pages, and databases. Text splitters chunk documents intelligently.
Vector store wrappers interface with FAISS, Pinecone, and Chroma.
Chains compose retriever plus LLM plus prompt into one workflow.
Memory stores conversation history across multiple turns. LangCha

In [10]:
# Test with an out-of-scope question
# The retriever still returns something — but the LLM must be instructed
# to say 'I don't know' when the retrieved content is not relevant
_ = show_retrieval("What is the best programming language for games?")

Query : 'What is the best programming language for games?'

Result 1  [Source: ui_frameworks.txt]
────────────────────────────────────────
APIs because it is fast and auto-documented. For teaching and demos
use Gradio. For internal tools use Streamlit. For production APIs
use FastAPI.

Result 2  [Source: ui_frameworks.txt]
────────────────────────────────────────
Streamlit is a Python framework for building interactive web
applications quickly. It is widely used for LLM chatbot interfaces
and data dashboards. Key features include st.chat_message for chat
UI and st.session_state for history. Gradio is similar but focused
on ML demos and sharing public links instantly. FastAPI is the
choice for production REST APIs because it is fast and auto-
documented. For

Result 3  [Source: langchain.txt]
────────────────────────────────────────
LangChain is an open-source Python framework for building LLM-
powered applications. It provides document loaders to load PDFs, web
pages, and databases. Te

## Step 6 — The LLM (Free, Local, No API Key)

We build a simple **mock LLM** that simulates generation by extracting the key sentence from the retrieved context.  
This runs **instantly** with zero downloads and zero errors — perfect for demonstrating the RAG pipeline to students.

> 💡 In a real project you would swap this for OpenAI, Claude, or a HuggingFace model.  
> The rest of the pipeline (retriever, memory, prompt) stays **exactly the same**.

In [11]:
# A lightweight mock LLM — no downloads, no API key, no errors
# It reads the retrieved context and returns a clean, relevant answer
# This lets us demonstrate the full RAG pipeline instantly

class TeachingLLM(LLM):
    """
    A simple LLM for teaching that extracts the most relevant sentence
    from the retrieved context. Runs instantly, no API key needed.
    In a real application, replace this with OpenAI or any other LLM.
    """

    @property
    def _llm_type(self):
        return "teaching_llm"

    def _call(self, prompt: str, stop: Optional[List[str]] = None) -> str:
        # Extract the context block from the prompt
        if "Context:" in prompt and "Question:" in prompt:
            context_start = prompt.find("Context:") + len("Context:")
            context_end   = prompt.find("Question:")
            context        = prompt[context_start:context_end].strip()
            question       = prompt[prompt.find("Question:") + len("Question:"):].replace("Answer:", "").strip()

            # Find the sentence from the context most relevant to the question
            sentences   = [s.strip() for s in context.replace("\n", " ").split(".") if len(s.strip()) > 30]
            if not sentences:
                return "I do not have that information in my knowledge base."

            q_words  = set(question.lower().split())
            scored   = [(len(q_words & set(s.lower().split())), s) for s in sentences]
            scored.sort(reverse=True)
            best     = scored[0][1].strip()

            if scored[0][0] == 0:
                return "I do not have that information in my knowledge base."

            return f"Based on the retrieved documents: {best}."

        return "Please provide context and a question."


llm = TeachingLLM()
print("✓ Teaching LLM ready")
print("  No API key needed | No downloads | Runs instantly")
print()
print("In a real project, replace TeachingLLM with:")
print("  from langchain_openai import ChatOpenAI")
print("  llm = ChatOpenAI(model='gpt-4o', api_key='your-key')")

✓ Teaching LLM ready
  No API key needed | No downloads | Runs instantly

In a real project, replace TeachingLLM with:
  from langchain_openai import ChatOpenAI
  llm = ChatOpenAI(model='gpt-4o', api_key='your-key')


## Step 7 — The RAG Prompt Template

The prompt template is the bridge between retrieval and generation. It tells the LLM:
- Here is the context (retrieved chunks)
- Here is the question
- Answer **only** from the context — say "I don't know" if the answer is not there

In [12]:
# The prompt template structures how context and question are sent to the LLM
# {context} is replaced with retrieved chunks
# {question} is replaced with the user's query

RAG_PROMPT = PromptTemplate(
    input_variables=["context", "question"],
    template="""You are a helpful assistant. Use ONLY the context below to answer the question.
If the answer is not in the context, say: I do not have that information in my knowledge base.
Do not make up information.

Context:
{context}

Question: {question}

Answer:"""
)

print("✓ Prompt template defined")
print()
print("Template structure:")
print("  1. Instruction  — tells the LLM to only use the context")
print("  2. {context}    — filled with the retrieved document chunks")
print("  3. {question}   — filled with the user's question")
print("  4. 'Answer:'    — primes the LLM to respond")

✓ Prompt template defined

Template structure:
  1. Instruction  — tells the LLM to only use the context
  2. {context}    — filled with the retrieved document chunks
  3. {question}   — filled with the user's question
  4. 'Answer:'    — primes the LLM to respond


In [13]:
# Build the full RetrievalQA chain
# This connects: user question → retriever → top chunks → prompt → LLM → answer
# chain_type='stuff' means all retrieved chunks are put into one single prompt
# return_source_documents=True means the answer also tells us which docs were used

retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

rag_chain = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=retriever,
    return_source_documents=True,
    chain_type_kwargs={"prompt": RAG_PROMPT}
)

print("✓ RAG chain built")
print("  Flow: question → FAISS retriever → top 3 chunks → LLM → answer")

✓ RAG chain built
  Flow: question → FAISS retriever → top 3 chunks → LLM → answer


In [14]:
# Helper to ask a question and show the answer with sources
def ask(question):
    print(f"Q: {question}")
    print("─" * 60)
    result  = rag_chain.invoke({"query": question})
    answer  = result["result"].strip()
    sources = set(doc.metadata["source"] for doc in result["source_documents"])
    print(f"A: {answer}")
    print(f"\n   Sources used: {', '.join(sources)}")
    print()

ask("What is RAG and how does it reduce hallucinations?")

Q: What is RAG and how does it reduce hallucinations?
────────────────────────────────────────────────────────────
A: Based on the retrieved documents: RAG is especially useful when knowledge needs frequent updates or when working with private, domain-specific information.

   Sources used: rag_overview.txt, hallucinations.txt



In [15]:
ask("What is the difference between FAISS and Pinecone?")

Q: What is the difference between FAISS and Pinecone?
────────────────────────────────────────────────────────────
A: Based on the retrieved documents: Choosing the right vector database depends on scale, cost, and whether you need cloud management  LangChain is an open-source Python framework for building LLM-powered applications.

   Sources used: langchain.txt, vector_databases.txt



In [16]:
# Question NOT in our knowledge base — model should say it does not know
ask("What is the best food to eat before a marathon?")

Q: What is the best food to eat before a marathon?
────────────────────────────────────────────────────────────
A: Based on the retrieved documents: Streamlit is a Python framework for building interactive web applications quickly.

   Sources used: ui_frameworks.txt, embeddings.txt



---
# PART 3 — LangChain Components

## What is LangChain?

LangChain is the most popular framework for building LLM-powered applications. It provides ready-made building blocks:

| Component | What It Does |
|-----------|-------------|
| **Document Loaders** | Load text from PDFs, websites, CSV files, databases |
| **Text Splitters** | Chunk documents intelligently |
| **Embeddings** | Wrap any embedding model (OpenAI, HuggingFace, Cohere) |
| **Vector Stores** | Interface with FAISS, Pinecone, Chroma, Weaviate |
| **Chains** | Compose retriever + LLM + prompt into one workflow |
| **Memory** | Store conversation history across turns |
| **Agents** | LLMs that decide which tools to use dynamically |

In [17]:
# Document Loaders — reference code showing the API pattern
# These show how you would load REAL files in a production project

print("LANGCHAIN DOCUMENT LOADERS — REFERENCE PATTERNS")
print("─" * 55)
print("""
from langchain.document_loaders import (
    PyPDFLoader,       # Load PDF files
    TextLoader,        # Load .txt files
    WebBaseLoader,     # Load web pages
    CSVLoader,         # Load CSV files
    DirectoryLoader,   # Load all files in a folder
)

# Load a PDF (each page becomes one Document)
loader = PyPDFLoader("report.pdf")
pages  = loader.load()

# Load a web page
loader = WebBaseLoader("https://example.com/article")
docs   = loader.load()

# Load all .txt files in a folder
loader = DirectoryLoader("./docs", glob="**/*.txt", loader_cls=TextLoader)
docs   = loader.load()

# Every Document has:
#   doc.page_content  — the text
#   doc.metadata      — {source, page, date, ...}
""")

LANGCHAIN DOCUMENT LOADERS — REFERENCE PATTERNS
───────────────────────────────────────────────────────

from langchain.document_loaders import (
    PyPDFLoader,       # Load PDF files
    TextLoader,        # Load .txt files
    WebBaseLoader,     # Load web pages
    CSVLoader,         # Load CSV files
    DirectoryLoader,   # Load all files in a folder
)

# Load a PDF (each page becomes one Document)
loader = PyPDFLoader("report.pdf")
pages  = loader.load()

# Load a web page
loader = WebBaseLoader("https://example.com/article")
docs   = loader.load()

# Load all .txt files in a folder
loader = DirectoryLoader("./docs", glob="**/*.txt", loader_cls=TextLoader)
docs   = loader.load()

# Every Document has:
#   doc.page_content  — the text
#   doc.metadata      — {source, page, date, ...}



In [18]:
# Demonstrate the two most common splitter types side by side
from langchain.text_splitter import CharacterTextSplitter

sample = (
    "Artificial intelligence is transforming every industry. "
    "Machine learning allows systems to learn from data.\n\n"
    "Deep learning uses neural networks with many layers. "
    "This enables learning of complex patterns.\n\n"
    "Natural language processing focuses on text and speech."
)

fixed_splitter     = CharacterTextSplitter(chunk_size=120, chunk_overlap=20, separator="\n")
recursive_splitter = RecursiveCharacterTextSplitter(chunk_size=120, chunk_overlap=20)

print("FIXED-SIZE SPLITTER")
print("─" * 50)
for i, c in enumerate(fixed_splitter.split_text(sample)):
    print(f"  Chunk {i+1}: {repr(c[:80])}")

print()
print("RECURSIVE SPLITTER  ← recommended")
print("─" * 50)
for i, c in enumerate(recursive_splitter.split_text(sample)):
    print(f"  Chunk {i+1}: {repr(c[:80])}")

print()
print("RecursiveCharacterTextSplitter respects paragraph boundaries.")
print("Always use this as your default.")

FIXED-SIZE SPLITTER
──────────────────────────────────────────────────
  Chunk 1: 'Artificial intelligence is transforming every industry. Machine learning allows '
  Chunk 2: 'Deep learning uses neural networks with many layers. This enables learning of co'
  Chunk 3: 'Natural language processing focuses on text and speech.'

RECURSIVE SPLITTER  ← recommended
──────────────────────────────────────────────────
  Chunk 1: 'Artificial intelligence is transforming every industry. Machine learning allows '
  Chunk 2: 'Deep learning uses neural networks with many layers. This enables learning of co'
  Chunk 3: 'Natural language processing focuses on text and speech.'

RecursiveCharacterTextSplitter respects paragraph boundaries.
Always use this as your default.


---
# PART 4 — Conversational RAG with Memory

## The Problem Without Memory

Without memory, every question is treated independently:
- User: *"What is a vector database?"* — model answers ✓
- User: *"What are some examples of it?"* — model has **no idea** what "it" refers to!

## The Solution: ConversationBufferMemory

`ConversationBufferMemory` stores the full conversation history.  
The chain automatically uses previous turns to understand follow-up questions.

In [19]:
# Build a Conversational RAG Chain with Memory
# ConversationalRetrievalChain = RetrievalQA + conversation history
# The chain reformulates ambiguous follow-up questions before retrieval

memory = ConversationBufferMemory(
    memory_key="chat_history",
    return_messages=True,
    output_key="answer"
)

conv_chain = ConversationalRetrievalChain.from_llm(
    llm=llm,
    retriever=retriever,
    memory=memory,
    return_source_documents=True,
    verbose=False
)

print("✓ Conversational RAG chain built")
print("  Memory : ConversationBufferMemory (stores full history)")
print("  Handles: follow-up questions and context-dependent queries")

✓ Conversational RAG chain built
  Memory : ConversationBufferMemory (stores full history)
  Handles: follow-up questions and context-dependent queries


In [20]:
# Simulate a multi-turn conversation
def chat(question):
    print(f"👤 User: {question}")
    result  = conv_chain.invoke({"question": question})
    answer  = result["answer"].strip()
    sources = set(doc.metadata["source"] for doc in result["source_documents"])
    print(f"🤖 Bot : {answer}")
    print(f"   [Sources: {', '.join(sources)}]")
    print()

# Turn 1 — ask about vector databases
chat("What is a vector database?")

👤 User: What is a vector database?
🤖 Bot : Please provide context and a question.
   [Sources: rag_overview.txt, embeddings.txt, vector_databases.txt]



In [21]:
# Turn 2 — follow-up using 'it' — the chain uses history to understand the reference
chat("What are some examples of it?")

👤 User: What are some examples of it?
🤖 Bot : Please provide context and a question.
   [Sources: rag_overview.txt, hallucinations.txt, vector_databases.txt]



In [22]:
# Turn 3 — different topic
chat("How does LangChain help build RAG applications?")

👤 User: How does LangChain help build RAG applications?
🤖 Bot : Please provide context and a question.
   [Sources: rag_overview.txt, hallucinations.txt, vector_databases.txt]



In [23]:
# Inspect the conversation memory — show what the chain remembers
print("Conversation memory stored so far:")
print("─" * 50)
for msg in memory.chat_memory.messages:
    role = "User" if msg.type == "human" else "Bot "
    print(f"[{role}] {str(msg.content)[:100]}...")

Conversation memory stored so far:
──────────────────────────────────────────────────
[User] What is a vector database?...
[Bot ] Please provide context and a question....
[User] What are some examples of it?...
[Bot ] Please provide context and a question....
[User] How does LangChain help build RAG applications?...
[Bot ] Please provide context and a question....


In [25]:
# FastAPI backend — reference pattern
# This is how you expose your RAG chain as a REST API

print("FASTAPI BACKEND — save as api.py and run: uvicorn api:app --reload")
print("─" * 60)
print("""
from fastapi import FastAPI
from pydantic import BaseModel

app = FastAPI(title="RAG API")

class QuestionRequest(BaseModel):
    question: str
    session_id: str = "default"

class AnswerResponse(BaseModel):
    answer: str
    sources: list

@app.post("/ask", response_model=AnswerResponse)
async def ask_question(request: QuestionRequest):
    result  = rag_chain.invoke({"query": request.question})
    sources = [doc.metadata["source"] for doc in result["source_documents"]]
    return AnswerResponse(answer=result["result"], sources=sources)

@app.get("/health")
async def health_check():
    return {"status": "ok"}

# After running, visit http://localhost:8000/docs for auto-generated API docs
""")

FASTAPI BACKEND — save as api.py and run: uvicorn api:app --reload
────────────────────────────────────────────────────────────

from fastapi import FastAPI
from pydantic import BaseModel

app = FastAPI(title="RAG API")

class QuestionRequest(BaseModel):
    question: str
    session_id: str = "default"

class AnswerResponse(BaseModel):
    answer: str
    sources: list

@app.post("/ask", response_model=AnswerResponse)
async def ask_question(request: QuestionRequest):
    result  = rag_chain.invoke({"query": request.question})
    sources = [doc.metadata["source"] for doc in result["source_documents"]]
    return AnswerResponse(answer=result["result"], sources=sources)

@app.get("/health")
async def health_check():
    return {"status": "ok"}

# After running, visit http://localhost:8000/docs for auto-generated API docs



---
# PART 6 — Evaluation Metrics for RAG

RAG has two parts that can each fail — you must evaluate both:

| Metric | Measures | Want |
|--------|----------|------|
| **Hit Rate** | Was the right document retrieved at all? | High |
| **Precision@K** | Of K retrieved docs, how many are relevant? | High |
| **Faithfulness** | Does the answer stick to the retrieved context? | High |
| **Answer Relevancy** | Does the answer address the question? | High |

**RAGAS** is an open-source framework that automates all of these.

In [26]:
# Measure retrieval quality on a small test set
def evaluate_retrieval(query, expected_source, k=3):
    results  = vectorstore.similarity_search(query, k=k)
    sources  = [doc.metadata["source"] for doc in results]
    hit      = int(expected_source in sources)
    precision = sources.count(expected_source) / k
    return hit, precision

test_cases = [
    ("What is vector embedding?",               "embeddings.txt"),
    ("What is FAISS?",                          "vector_databases.txt"),
    ("How does LangChain handle conversation?", "langchain.txt"),
    ("What causes LLM hallucinations?",         "hallucinations.txt"),
    ("What is Gradio used for?",                "ui_frameworks.txt"),
]

print("RETRIEVAL EVALUATION")
print("═" * 60)
print(f"{'Query':<43} {'Hit':>5}  {'Precision@3':>12}")
print("─" * 60)

total_hit = 0
for query, expected in test_cases:
    hit, prec = evaluate_retrieval(query, expected)
    total_hit += hit
    icon = "✓" if hit else "✗"
    print(f"{query:<43} {icon:>5}  {prec:>12.2f}")

print("─" * 60)
print(f"Overall Hit Rate: {total_hit}/{len(test_cases)} = {total_hit/len(test_cases)*100:.0f}%")
print()
print("Hit Rate  = right document retrieved in top k")
print("Precision = fraction of retrieved docs that are relevant")

RETRIEVAL EVALUATION
════════════════════════════════════════════════════════════
Query                                         Hit   Precision@3
────────────────────────────────────────────────────────────
What is vector embedding?                       ✓          0.67
What is FAISS?                                  ✓          0.33
How does LangChain handle conversation?         ✓          0.67
What causes LLM hallucinations?                 ✓          0.33
What is Gradio used for?                        ✓          0.67
────────────────────────────────────────────────────────────
Overall Hit Rate: 5/5 = 100%

Hit Rate  = right document retrieved in top k
Precision = fraction of retrieved docs that are relevant


In [27]:
# Demonstrate how chunk size affects the result
long_doc = Document(
    page_content=(
        "RAG was introduced by Lewis et al. in 2020. "
        "It combines a dense retriever with a sequence-to-sequence model. "
        "The retriever finds relevant passages from a large corpus. "
        "The generator produces an answer conditioned on those passages. "
        "This reduces hallucinations significantly. "
        "It allows the model to cite its sources."
    ),
    metadata={"source": "rag_paper.txt"}
)

for size, overlap in [(100, 10), (300, 50)]:
    sp = RecursiveCharacterTextSplitter(chunk_size=size, chunk_overlap=overlap)
    c  = sp.split_documents([long_doc])
    print(f"chunk_size={size}, overlap={overlap}  →  {len(c)} chunk(s)")
    for i, ch in enumerate(c):
        print(f"  Chunk {i+1} ({len(ch.page_content)} chars): {ch.page_content[:70]}...")
    print()

print("Smaller chunks = more precise retrieval but may miss context.")
print("Larger chunks  = more context but retrieval may be less precise.")
print("Start with 400–600 chars and tune based on your evaluation results.")

chunk_size=100, overlap=10  →  4 chunk(s)
  Chunk 1 (80 chars): RAG was introduced by Lewis et al. in 2020. It combines a dense retrie...
  Chunk 2 (97 chars): with a sequence-to-sequence model. The retriever finds relevant passag...
  Chunk 3 (91 chars): The generator produces an answer conditioned on those passages. This r...
  Chunk 4 (55 chars): significantly. It allows the model to cite its sources....

chunk_size=300, overlap=50  →  2 chunk(s)
  Chunk 1 (297 chars): RAG was introduced by Lewis et al. in 2020. It combines a dense retrie...
  Chunk 2 (55 chars): significantly. It allows the model to cite its sources....

Smaller chunks = more precise retrieval but may miss context.
Larger chunks  = more context but retrieval may be less precise.
Start with 400–600 chars and tune based on your evaluation results.
